In [4]:
import json
from pathlib import Path
from pprint import pprint
from uuid import uuid4

from src.context.contextbuilder import ContextBuilder
from src.engine.LlmProviderManager import LlmProvider
from src.models.MemoryEvent import MemoryEvent
from src.models.ToolResult import ToolResult




def load_config() -> dict:
    with open(
"config.json",
        "r",
        encoding="utf-8",
    ) as f:
        return json.load(f)


def make_event(
    content: str,
    event_type: str,
    source: str,
    step: int,
    metadata: dict | None = None,
) -> MemoryEvent:

    return MemoryEvent(
        id=uuid4(),
        event_type=event_type,
        content=content,
        source=source,
        step=step,
        metadata=metadata or {},
    )


def print_separator(title: str) -> None:
    print()
    print("=" * 100)
    print(title)
    print("=" * 100)


def print_prompt(
    messages: list[dict],
) -> None:

    print_separator("FINAL CONTEXT PROMPT")

    for index, message in enumerate(messages):

        print()
        print(f"---------------- MESSAGE {index} ----------------")

        print(f"ROLE: {message.get('role')}")

        print("CONTENT:")

        print(
            message.get(
                "content",
                "",
            )
        )


def test_normal_context() -> None:

    print_separator("TEST 1 — NORMAL CONTEXT")

    config = load_config()

    llm = LlmProvider(config)

    builder = ContextBuilder(
        config=config,
        llm_provider=llm,
    )

    tool_stdout = (
        "from flask import Flask\n"
        "\n"
        "app = Flask(__name__)\n"
        "\n"
        "@app.route('/')\n"
        "def home():\n"
        "    return 'Shop'\n"
    )

    events = [
        make_event(
            content=("Build a Flask shop website."),
            event_type="user_input",
            source="user",
            step=1,
        ),
        make_event(
            content=("I will inspect the project " "structure first."),
            event_type="assistant_message",
            source="assistant",
            step=2,
        ),
        make_event(
            content=json.dumps(
                {
                    "tool_name": "Read",
                    "arguments": {
                        "path": "shop/app.py",
                    },
                }
            ),
            event_type="tool_call",
            source="assistant",
            step=3,
        ),
        make_event(
            content=tool_stdout,
            event_type="tool_result",
            source="tool",
            step=4,
        ),
    ]

    tool_result = ToolResult(
        success=True,
        name="Read",
        content={
            "stdout": tool_stdout,
            "path": "shop/app.py",
        },
        metadata={
            "call_id": "call-001",
            "exit_code": 0,
            "duration": 0.12,
            "retryable": False,
        },
    )

    task = {
        "goal": "Finish the shop implementation",
        "requirements": [
            "Use Flask",
            "Keep existing routes",
            "Add product images",
        ],
    }

    active_skills = {
        "filesystem": {
            "enabled": True,
        },
        "python": {
            "enabled": True,
        },
    }

    agent_state = {
        "iteration": 4,
        "status": "working",
    }

    progress = {
        "completed": [
            "Inspected project structure",
            "Inspected shop/app.py",
        ],
        "pending": [
            "Modify shop/app.py",
            "Add product images",
            "Run tests",
        ],
        "failed": [],
    }

    messages = builder.build_context(
        events=events,
        tool_result=tool_result,
        task=task,
        active_skills=active_skills,
        agent_state=agent_state,
        progress=progress,
    )

    print_prompt(messages)

    # ---------------------------------------------------------
    # Basic contract
    # ---------------------------------------------------------

    assert isinstance(messages, list)

    assert messages

    for message in messages:
        assert isinstance(message, dict)

        assert "role" in message
        assert "content" in message

    # ---------------------------------------------------------
    # Exactly one system message
    # ---------------------------------------------------------

    system_messages = [
        message for message in messages if message.get("role") == "system"
    ]

    assert len(system_messages) == 1

    system_prompt = system_messages[0]["content"]

    # ---------------------------------------------------------
    # Required sections
    # ---------------------------------------------------------

    required_sections = [
        "<task>",
        "<active_skills>",
        "<agent_state>",
        "<progress>",
        "<last_action>",
        "<last_observation>",
        "<runtime>",
    ]

    for section in required_sections:
        assert section in system_prompt

    # ---------------------------------------------------------
    # Runtime
    # ---------------------------------------------------------

    assert '"os"' in system_prompt

    # ---------------------------------------------------------
    # Task
    # ---------------------------------------------------------

    assert "Finish the shop implementation" in system_prompt

    assert "Add product images" in system_prompt

    # ---------------------------------------------------------
    # Skills
    # ---------------------------------------------------------

    assert "filesystem" in system_prompt

    assert "python" in system_prompt

    # ---------------------------------------------------------
    # Agent state
    # ---------------------------------------------------------

    assert '"iteration": 4' in system_prompt

    # ---------------------------------------------------------
    # Progress
    # ---------------------------------------------------------

    assert "Inspected shop/app.py" in system_prompt

    assert "Add product images" in system_prompt

    # ---------------------------------------------------------
    # Last action
    # ---------------------------------------------------------

    assert '"event_type": "tool_call"' in system_prompt

    assert '"path": "shop/app.py"' in system_prompt

    # ---------------------------------------------------------
    # Last observation
    # ---------------------------------------------------------

    assert '"name": "Read"' in system_prompt

    assert '"success": true' in system_prompt

    assert '"exit_code": 0' in system_prompt

    # ---------------------------------------------------------
    # Tool output
    # ---------------------------------------------------------

    serialized_tool_output = json.dumps(
        tool_stdout,
        ensure_ascii=False,
    )

    assert serialized_tool_output in system_prompt

    # Must appear exactly once.
    assert system_prompt.count(serialized_tool_output) == 1

    # ---------------------------------------------------------
    # Tool output must NOT be conversation
    # ---------------------------------------------------------

    conversation_messages = [
        message
        for message in messages
        if message.get("role")
        in {
            "user",
            "assistant",
        }
    ]

    for message in conversation_messages:
        assert tool_stdout not in str(
            message.get(
                "content",
                "",
            )
        )

    # ---------------------------------------------------------
    # Conversation
    # ---------------------------------------------------------

    assert any(
        message.get("role") == "user"
        and message.get("content") == "Build a Flask shop website."
        for message in messages
    )

    assert any(
        message.get("role") == "assistant"
        and message.get("content") == "I will inspect the project structure first."
        for message in messages
    )

    # tool_call must NOT become conversation.
    assert not any(
        message.get("role") == "assistant"
        and '"tool_name": "Read"'
        in str(
            message.get(
                "content",
                "",
            )
        )
        for message in messages
    )

    # ---------------------------------------------------------
    # State objects
    # ---------------------------------------------------------

    assert builder.window.task == task

    assert builder.window.active_skills == active_skills

    assert builder.window.agent_state == agent_state

    assert builder.window.progress == progress

    assert builder.window.last_action["event_type"] == "tool_call"

    observation = builder.window.last_observation

    assert observation["name"] == "Read"

    assert observation["success"] is True

    assert observation["content"]["stdout"] == tool_stdout

    assert observation["metadata"]["call_id"] == "call-001"

    # ---------------------------------------------------------
    # Token budget
    # ---------------------------------------------------------

    estimated_tokens = builder.tokenbudget.estimate_messages_tokens(messages)

    print_separator("TOKEN ANALYSIS")

    print(f"Estimated tokens: {estimated_tokens}")

    print(f"Budget: {builder.tokenbudget.budget}")

    print(f"Remaining: " f"{builder.tokenbudget.remaining_tokens(messages)}")

    assert estimated_tokens <= builder.tokenbudget.budget

    print("\n✅ NORMAL CONTEXT TEST PASSED")


def test_contextbuilder_overflow() -> None:

    print_separator("TEST 2 — CONTEXT OVERFLOW")

    config = load_config()

    llm = LlmProvider(config)

    builder = ContextBuilder(
        config=config,
        llm_provider=llm,
    )

    # Make the context very large.
    huge_text = "This is old conversation data. " * 5000

    events = [
        make_event(
            content=huge_text,
            event_type="user_input",
            source="user",
            step=1,
        ),
    ]

    tool_result = ToolResult(
        success=True,
        name="Read",
        content={
            "stdout": "final observation",
            "path": "shop/app.py",
        },
        metadata={
            "call_id": "overflow-test",
            "exit_code": 0,
        },
    )

    # ---------------------------------------------------------
    # Stub compactor
    # ---------------------------------------------------------

    compact_called = {
        "value": False,
    }

    def fake_compact(
        context: str,
        max_length: int,
    ) -> str:

        compact_called["value"] = True

        return (
            "Previous conversation compacted. "
            "The project is a Flask shop. "
            "Continue implementation."
        )

    builder.compactor.compact = fake_compact

    messages = builder.build_context(
        events=events,
        tool_result=tool_result,
        task={"goal": "Continue the shop implementation"},
        active_skills={
            "filesystem": {
                "enabled": True,
            }
        },
        agent_state={
            "iteration": 20,
            "status": "working",
        },
        progress={
            "completed": ["Project inspected"],
            "pending": ["Continue implementation"],
            "failed": [],
        },
    )

    print_prompt(messages)

    final_tokens = builder.tokenbudget.estimate_messages_tokens(messages)

    print_separator("OVERFLOW TOKEN ANALYSIS")

    print(f"Final tokens: {final_tokens}")

    print(f"Budget: {builder.tokenbudget.budget}")

    print(f"Compactor called: " f"{compact_called['value']}")

    # ---------------------------------------------------------
    # Assertions
    # ---------------------------------------------------------

    assert compact_called["value"] is True

    assert final_tokens <= builder.tokenbudget.budget

    # Compacted context must exist.
    assert any(
        "<compacted_conversation>"
        in str(
            message.get(
                "content",
                "",
            )
        )
        for message in messages
    )

    # One system message only.
    system_messages = [
        message for message in messages if message.get("role") == "system"
    ]

    assert len(system_messages) == 1

    # Last observation must survive compaction.
    system_prompt = system_messages[0]["content"]

    assert "final observation" in system_prompt

    print("\n✅ OVERFLOW TEST PASSED")


def main() -> None:

    test_normal_context()

    test_contextbuilder_overflow()

    print_separator("ALL CONTEXT BUILDER TESTS")

    print("🎉 ALL TESTS PASSED")


if __name__ == "__main__":
    main()

[17:19:48] [INFO] [[PROVIDERREGISTRY]] → Found provider package: ollama
[17:19:48] [INFO] [[PROVIDERREGISTRY]] → Loaded module: <module 'src.engine.providers.builtin.ollama' from '/home/itsnxfi3/Desktop/Evana-agent-runtime/src/engine/providers/builtin/ollama/__init__.py'>
[17:19:48] [INFO] [[PROVIDERREGISTRY]] → Exports: ['OllamaProvider']
[17:19:48] [INFO] [[PROVIDERREGISTRY]] → Found provider class: <class 'src.engine.providers.builtin.ollama.ollama.OllamaProvider'>
[17:19:48] [INFO] [[PROVIDERREGISTRY]] → Instantiated: ollama
[17:19:48] [INFO] [[PROVIDERREGISTRY]] → Discovered 1 provider(s)
[17:19:48] [INFO] [[LLM]] → Loading ollama provider
[17:19:48] [INFO] [[TOKENBUDGET]] → Context length=120000, budget=103616, safe_margin=16384, compaction_target=16384, chars_per_token=4.0
[17:19:48] [INFO] [[PROVIDERREGISTRY]] → Found provider package: ollama
[17:19:48] [INFO] [[PROVIDERREGISTRY]] → Loaded module: <module 'src.engine.providers.builtin.ollama' from '/home/itsnxfi3/Desktop/Evana-


TEST 1 — NORMAL CONTEXT

FINAL CONTEXT PROMPT

---------------- MESSAGE 0 ----------------
ROLE: system
CONTENT:
<task>
{
  "goal": "Finish the shop implementation",
  "requirements": [
    "Use Flask",
    "Keep existing routes",
    "Add product images"
  ]
}
</task>

<active_skills>
{
  "filesystem": {
    "enabled": true
  },
  "python": {
    "enabled": true
  }
}
</active_skills>

<agent_state>
{
  "iteration": 4,
  "status": "working"
}
</agent_state>

<progress>
{
  "completed": [
    "Inspected project structure",
    "Inspected shop/app.py"
  ],
  "pending": [
    "Modify shop/app.py",
    "Add product images",
    "Run tests"
  ],
  "failed": []
}
</progress>

<last_action>
{
  "event_type": "tool_call",
  "source": "assistant",
  "content": "{\"tool_name\": \"Read\", \"arguments\": {\"path\": \"shop/app.py\"}}",
  "step": 3,
  "timestamp": "2026-09-16 17:19:48.727287"
}
</last_action>

<last_observation>
{
  "success": true,
  "name": "Read",
  "content": {
    "stdout": "

AssertionError: 

In [ ]:
context